# Lab RAG on a real company handbook

**(RAG: retrieval, embeddings and evaluation)**

GitLab runs the whole company from a **public handbook** ([handbook.gitlab.com](https://handbook.gitlab.com), CC BY-SA 4.0): thousands of pages of HR, finance, security and legal policy. We take **38 of those pages** as a stand-in for *your* organization's documents and build an assistant that answers **from the documents, with sources**, says *"I don't know"* when they don't cover the question — and we **measure** which chunking strategy finds the answers, and see what metadata filters do.

```
index once:   pages → clean → chunk → embed → ChromaDB (+ metadata)
per question: question → [filter on metadata] → top-k chunks → similarity gate → prompt → cited answer
```

The three exercises are **prompts**: you replace the text that says *Write your prompt here*; all the code around them is given. The Extras after the summary (overlap, k, keyword search, your own documents) are optional.

### Setting up the LLM

In [1]:
!pip install -q -U transformers accelerate chromadb pypdf sentence-transformers
import torch, json, re
from transformers import AutoTokenizer, AutoModelForImageTextToText

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.

In [2]:
#@title Load Qwen3.5-4B (the same model in every lab)
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen3.5-4B"                     # about 9 GB in float16: fits a free Colab T4
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto")
model.eval()
print("running on:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

running on: Tesla T4


In [3]:
#@title chat(): the one function that talks to the model (the same in every lab)
def chat(messages, system=None, max_new_tokens=400, temperature=0.0, thinking=False, tools=None):
    """A prompt string or a list of {"role", "content"} messages -> the model's reply as text.
    system: an optional system prompt, put in front of the messages.
    temperature=0 is greedy (the same reply every time); above 0 it samples, with top_p=0.8 and top_k=20
    from the Qwen3.5 model card. thinking=True lets the model reason in <think>...</think> before answering.
    tools: tool schemas for native tool calling (the reply then may contain <tool_call> blocks)."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if system:
        messages = [{"role": "system", "content": system}] + messages
    prompt = tok.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True,
                                     enable_thinking=thinking)                 # wrap in the model's own chat markup
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    sampling = dict(do_sample=True, temperature=temperature, top_p=0.8, top_k=20) if temperature > 0 else dict(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **sampling)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()  # keep only the new text

In [4]:
#@title chat_json(): chat() that returns a dict
import json

def chat_json(messages, **kw):
    """chat(), but returns the first JSON object in the reply as a dict ({} if there is none).
    Small models sometimes write broken JSON: on a parse error, show the model the error and ask once more."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    for attempt in range(2):
        text = chat(messages, **kw)
        start = text.find("{")
        if start < 0:
            return {}
        try:
            return json.JSONDecoder().raw_decode(text[start:])[0]             # parse one object, ignore what follows
        except json.JSONDecodeError as err:
            messages = messages + [{"role": "assistant", "content": text},
                                   {"role": "user", "content": f"That JSON is invalid ({err}). Reply with only the corrected JSON."}]
    return {}

In [5]:
#@title chat_batch(): chat() for many prompts at once
def chat_batch(prompts, system=None, max_new_tokens=400, temperature=0.0, thinking=False, batch_size=16):
    """Same settings as chat(), but one generate() call per batch_size prompts: much faster on a GPU."""
    conversations = [[{"role": "user", "content": p}] if isinstance(p, str) else p for p in prompts]
    if system:
        conversations = [[{"role": "system", "content": system}] + c for c in conversations]
    texts = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
             for c in conversations]
    tok.padding_side = "left"                     # pad on the left, so every reply starts right after its prompt
    sampling = dict(do_sample=True, temperature=temperature, top_p=0.8, top_k=20) if temperature > 0 else dict(do_sample=False)
    replies = []
    for i in range(0, len(texts), batch_size):
        inputs = tok(texts[i:i + batch_size], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **sampling)
        replies += tok.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return [r.strip() for r in replies]

In [ ]:
print(chat("What is it like to be an LLM? in one sentence"))

## 0 The problem: the model has never read your documents

Even when it *has* seen a public page during training, you cannot tell a memory from a guess — and it has certainly never seen your internal ones.

In [6]:
print(chat("At GitLab, what is the daily limit for meals while traveling for work? Answer in two sentences.",
           max_new_tokens=150))                                  # a guess, a shrug, or an unverifiable "fact"

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


GitLab does not have a single, universal daily limit for meals while traveling for work, as reimbursement policies vary significantly by country, region, and specific project requirements. Instead, employees typically follow the local government's official per diem rates or the specific guidelines set by their local office or project lead to determine eligible meal expenses.


In our test run the model said GitLab has no single daily limit and pointed to per diem rates — fluent, confident and wrong: the handbook sets **$120 USD per day**. Remember the question; it comes back in sections 5 and 7.

## 1 Load the documents — with metadata

Real RAG starts with pulling documents out of a source system (a wiki, SharePoint, a ticketing tool). Here the source is GitLab's handbook repository. For each page we keep **metadata** next to the text: `department`, `country`, `updated` date and `url`. Text answers *what does it say*; metadata answers *which documents may even be considered* — you will see why in section 6.

In [7]:
import requests, re, pandas as pd, concurrent.futures as cf
from urllib.parse import quote

API = "https://gitlab.com/api/v4/projects/gitlab-com%2Fcontent-sites%2Fhandbook/repository"
REF = "df66e66b937d38c1ed4e3dd452927ddf01be58b0"      # one fixed commit of the handbook, so everyone sees the same text
PAGES = """
people-group/time-off-and-absence/time-off-types.md
people-group/time-off-and-absence/leave-types.md
people-group/anti-harassment.md
people-group/contracts-probation-periods.md
people-group/acceptable-use-policy.md
people-group/general-onboarding/_index.md
people-group/offboarding/faq.md
people-policies/inc-usa.md
people-policies/leave-of-absence/us.md
people-policies/ireland-ltd/right-to-disconnect-policy.md
people-policies/france-sas/remote-work-charter.md
total-rewards/benefits/_index.md
total-rewards/benefits/parental-leave-toolkit.md
total-rewards/benefits/general-and-entity-benefits/ltd-benefits-uk.md
total-rewards/benefits/general-and-entity-benefits/gitlab-ireland-ltd.md
total-rewards/benefits/general-and-entity-benefits/bv-benefits-netherlands.md
total-rewards/benefits/general-and-entity-benefits/inc-benefits-us/_index.md
total-rewards/benefits/general-and-entity-benefits/pty-benefits-australia.md
total-rewards/benefits/general-and-entity-benefits/canada-corp-benefits.md
total-rewards/compensation/_index.md
total-rewards/stock-options.md
finance/expenses.md
finance/spending-company-money.md
finance/procurement/individual-use-software.md
finance/accounts-payable/corp-credit-cards.md
security/policies_and_standards/password-standard.md
security/policies_and_standards/data-classification-standard.md
security/policies_and_standards/physical-security-standard-for-company-assets.md
security/policies_and_standards/records-retention-deletion.md
security/policies_and_standards/token-management-standard.md
legal/anti-corruption-policy.md
legal/anti-fraud-policy.md
legal/anti-retaliation-policy.md
legal/policies/gifts-contributions.md
legal/acceptable-use-policy.md
business-technology/_index.md
communication/confidentiality-levels.md
hiring/referral-process.md
""".split()

# Metadata we can derive from the path itself: the folder gives the department, a keyword gives the country.
DEPARTMENT = {"people-group": "People", "people-policies": "People", "total-rewards": "Total Rewards", "finance": "Finance", "security": "Security", "legal": "Legal", "business-technology": "IT", "communication": "Communication", "hiring": "Hiring"}
COUNTRY = {"benefits-uk": "UK", "ireland": "IE", "netherlands": "NL", "benefits-us": "US", "inc-usa": "US", "/us.md": "US", "india": "IN", "france": "FR", "australia": "AU", "canada": "CA"}

def fetch(page):
    """One handbook page -> {title, department, country, updated, url, text}. The metadata is what filters will use later."""
    path = "content/handbook/" + page
    text = requests.get(f"{API}/files/{quote(path, safe='')}/raw", params={"ref": REF}).text      # raw markdown at that commit
    updated = requests.get(f"{API}/commits", params={"path": path, "ref_name": REF, "per_page": 1}).json()[0]["committed_date"][:10]   # last edit date = freshness
    front, body = text.split("\n---\n", 1)                                      # YAML front matter | markdown body
    return {"title": re.search(r'^title:\s*"?(.*?)"?\s*$', front, re.M).group(1),   # title lives in the front matter
            "department": DEPARTMENT[page.split("/")[0]],
            "country": next((c for key, c in COUNTRY.items() if key in page), "global"),   # no keyword matched -> applies everywhere
            "updated": int(updated.replace("-", "")),                          # 20260615 -> numbers can be compared in filters
            "url": "https://handbook.gitlab.com/handbook/" + page.replace("/_index.md", "/").replace(".md", "/"),   # real link, for citations
            "text": body}

with cf.ThreadPoolExecutor(8) as pool:                                          # 8 pages at a time, ~15 seconds
    docs = list(pool.map(fetch, PAGES))
print(len(docs), "pages,", sum(len(d["text"]) for d in docs), "characters")
pd.DataFrame(docs).drop(columns="text").sample(8, random_state=1)               # look at the metadata, not the text

38 pages, 714794 characters


,title,department,country,updated,url
2,Anti-Harassment Policy,People,global,20260206,https://handbook.gitlab.com/handbook/people-gr...
29,GitLab Token Management Standard,Security,global,20260212,https://handbook.gitlab.com/handbook/security/...
3,"Contracts, Background Screenings, Probation Pe...",People,global,20260619,https://handbook.gitlab.com/handbook/people-gr...
22,Spending Company Money,Finance,global,20250130,https://handbook.gitlab.com/handbook/finance/s...
25,GitLab Password Standards,Security,global,20260728,https://handbook.gitlab.com/handbook/security/...
27,Physical Security Standard for Company Assets,Security,global,20260212,https://handbook.gitlab.com/handbook/security/...
21,Global Travel and Expense Policy,Finance,global,20260831,https://handbook.gitlab.com/handbook/finance/e...
35,IT Enterprise Applications,IT,global,20250117,https://handbook.gitlab.com/handbook/business-...


## 2 Clean

Raw pages carry markup that has nothing to do with meaning (template shortcodes, HTML tags, link URLs). Embedding noise wastes the small context we can give the model, so strip it.

In [8]:
def clean(text):
    text = re.sub(r"\{\{[<%].*?[>%]\}\}|<!--.*?-->|<[^>\n]+>", "", text, flags=re.S)    # template shortcodes, HTML comments and tags
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)                                 # [link text](url) -> link text
    return re.sub(r"[ \t]+", " ", text).strip() # collapse runs of spaces

for d in docs:
    d["text"] = clean(d["text"])
print(docs[0]["title"], "\n", docs[0]["text"][:1000])

Time Off Types 
 For information specific to your location, please review this page.

## Flexible Paid Time Off (PTO)

*This page focuses specifically on GitLab's Flexible PTO policy and is written with a focus on using PTO to fulfill personal obligations, rest or recharge, enjoy a vacation, or spend time with family and friends. You can learn more about GitLab's overall Philosophy to Time Away from Work here and find more detailed policies on all Time Off types and Leave types in their respective handbook sections.*

### Overview

GitLab offers a Flexible PTO policy that empowers team members to take the time they need while ensuring business continuity. This policy reflects our values of transparency, collaboration, and GitLab's overall Philosophy to Time Away from Work.

- Our Flexible PTO policy means that as long as you have coordinated with your manager about your planned time off and made a coverage plan, you are free to take PTO.
- GitLab does not set a specific limit on the am

## 3 Chunking — four strategies

A page is too long to embed as one vector (one vector cannot mean 30 different things), so we split it. *How* we split is a real design decision:

| Strategy | Idea | Risk |
|---|---|---|
| **tiny** — fixed 100 chars | very precise matching | the answer gets cut in half |
| **medium** — fixed 300 chars | a few sentences per chunk | may split a table or list |
| **huge** — fixed 3,000 chars | lots of context per hit | one vector for many topics: similarity gets diluted |
| **sections** — split on headings, cap at 800 | follows the author's structure, heading travels with the text | sections vary wildly in length |

In [9]:
def chunk_fixed(text, size=300):
    """Split into pieces of about `size` characters, on word boundaries."""
    pieces, current = [], ""
    for word in text.split():
        if len(current) + len(word) > size:        # `size` is a NUMBER of characters, not a chunker function
            pieces.append(current.strip()); current = ""     # close the chunk, start a new one
        current += word + " "
    return pieces + [current.strip()]              # don't lose the last, partial chunk

def chunk_sections(text, max_size=800):
    """Split on markdown headings; keep the heading with its text; cut very long sections with chunk_fixed."""
    pieces = []
    for section in re.split(r"\n(?=#{1,4} )", text):         # lookahead: split before a heading, keep the heading
        heading = section.split("\n")[0] if section.startswith("#") else ""
        pieces += [p if p.startswith(heading) else heading + "\n" + p for p in chunk_fixed(section, max_size)]   # heading travels with every piece
    return [p for p in pieces if len(p) > 60]                # drop scraps too short to mean anything

In [10]:
# Each value is a one-argument chunker, so the indexing loop below can treat them all the same.
STRATEGIES = {"tiny": lambda t: chunk_fixed(t, 100), "medium": lambda t: chunk_fixed(t, 300),
              "huge": lambda t: chunk_fixed(t, 3000), "sections": chunk_sections}


In [12]:
page = next(d for d in docs if d["title"] == "GitLab Password Standards")       # one page, four ways to cut it
for name, chunker in STRATEGIES.items():
    pieces = chunker(page["text"])
    print(f"{name:<9} {len(pieces):>3} chunks | first chunk: {pieces[0][:500]!r}")

tiny       86 chunks | first chunk: '## Purpose This document outlines information security password standards intended to protect GitLab'
medium     29 chunks | first chunk: '## Purpose This document outlines information security password standards intended to protect GitLab information systems and other resources containing confidential (Red and Orange) GitLab data from unauthorized use, where technically feasible. ## Scope Applies to all GitLab team members,'
huge        3 chunks | first chunk: '## Purpose This document outlines information security password standards intended to protect GitLab information systems and other resources containing confidential (Red and Orange) GitLab data from unauthorized use, where technically feasible. ## Scope Applies to all GitLab team members, contractors, advisors, and contracted parties interacting with GitLab computing resources and accessing confidential data. ## Roles & Responsibilities | Role | Responsibility | |-----------|-----------| | G

## 4 Embed and index — one collection per strategy

Each chunk becomes a vector with a small embedding model (`all-MiniLM-L6-v2`, 22M parameters, runs on the GPU in seconds) and is stored in **ChromaDB** together with its metadata. Same meaning → nearby vectors, even with different words.

In [13]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedder = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2", device="cuda")   # text -> 384-dim vector, on the GPU
client = chromadb.Client()                                                                      # in-memory DB: gone when the runtime restarts

def index(name, chunker):
    kb = client.create_collection(name, embedding_function=embedder, metadata={"hnsw:space": "cosine"})   # cosine: compare direction, not length
    for n, d in enumerate(docs):
        pieces = chunker(d["text"])
        kb.add(ids=[f"{n}-{i}" for i in range(len(pieces))], documents=pieces,        # ids must be unique within the collection
               metadatas=[{k: d[k] for k in ("title", "department", "country", "updated", "url")}] * len(pieces))   # every chunk inherits its page's metadata
    return kb

KBS = {name: index(name, chunker) for name, chunker in STRATEGIES.items()}     # four collections, same documents, different cuts
{name: kb.count() for name, kb in KBS.items()}

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'tiny': 6412, 'medium': 2115, 'huge': 226, 'sections': 1160}

## 5 Search by meaning — and measure which chunking works

`search()` embeds the question and returns the k nearest chunks with a similarity score (1 = identical meaning). First look at one question across the four strategies, then **measure**: fourteen questions whose answers we know; does the answer text appear in the top-3 chunks?

In [14]:
def search(kb, question, k=3, where=None):
    """[(similarity, metadata, chunk_text), ...] for the k most similar chunks; `where` filters on metadata first."""
    r = kb.query(query_texts=[question], n_results=k, where=where)    # the question is embedded with the same model as the chunks
    return [(round(1 - dist, 2), meta, doc) for doc, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

In [15]:
q = "What is the daily limit for meals while traveling for work?"
for name, kb in KBS.items():
    score, meta, text = search(kb, q, 1)[0]
    print(f"{name:<9} {score}  {meta['title'][:30]:<30} | {text[:300]!r}")

tiny      0.67  Global Travel and Expense Poli | 'lodging and meals during the part of the trip that is work-related. Depending on the distance of'
medium    0.56  Global Travel and Expense Poli | 'representative (including any government prospects and government customers). - Meals are otherwise reimbursable, however Team Members should be mindful of budget while claiming meals with clients and partners. - $80 limit per person per meal. - All Participants should be added (both internal and'
huge      0.34  Time Off Types                 | 'Workday, please refer to: Workday: Single Source of Truth for Absence. ### Frequently Asked Public Holiday Questions ***Help! I’m receiving an error message, what does this mean?*** - All team members will see public holiday balance accrual in Workday. If you encounter an error when trying to submit'
sections  0.62  Global Travel and Expense Poli | '### Meals: Company Provided while Traveling - Meals while traveling on Company business are reimbursa

Only **sections** puts the rule first: its chunk starts at the heading *Meals: Company Provided while Traveling* and keeps the sentence whole (*up to $120 USD ... per day*). In our test run medium's best chunk was about meals **with clients** ($80 per person per meal), and tiny's and huge's best chunks did not contain a limit at all. One question proves nothing, so now the fourteen:

In [16]:
EVAL = [  # (question, text that must appear in the retrieved context) -- specific, so only the real answer matches
    ("How many days of time off should I take at minimum each year?", "minimum 25 days"),
    ("What percentage of my salary does the company pay into the UK pension?", ["GitLab: 4%", "GitLab contributes 4%"]),  # either sentence
    ("Which private medical insurance provider covers team members in Ireland?", "through VHI"),
    ("What is the daily limit for meals while traveling for work?", "$120 USD"),
    ("How many weeks of paid parental leave do team members get?", "16 weeks"),
    ("What is the minimum password length?", "12 characters"),
    ("How long can a personal access token be valid before it expires?", "Personal Access Tokens - 30 days"),
    ("Can I install Chrome extensions on my work laptop?", "Chrome Extensions"),
    ("What is the maximum value of a gift to a US government official per event?", "$20 USD"),
    ("How do I return my laptop when I leave?", "prepaid shipping label"),
    ("How long are production change records retained?", "Production changes | 3 years"),
    ("Are interns eligible for referral bonuses?", "not eligible for referral bonuses"),
    ("How many vacation days are Netherlands team members entitled to?", "20 vacation days"),
    ("What is the reimbursement limit for internet while traveling?", "up to $50 USD"),
]

In [17]:
def has_answer(text, ans):
    "Does the chunk contain the answer text (or one of a list of answer texts)? Ignores case."
    return any(a.lower() in text.lower() for a in ([ans] if isinstance(ans, str) else ans))

def evaluate(kb, k=3):
    # This measures RETRIEVAL only — the model is not involved yet. If the answer text isn't here, no prompt can rescue it.
    found = sum(any(has_answer(text, ans) for _, _, text in search(kb, q, k)) for q, ans in EVAL)
    avg = sum(len(c) for c in kb.get()["documents"]) // kb.count()       # average chunk size
    return {"chunks": kb.count(), "avg chars": avg, "context per question": avg * k,   # what each strategy costs in prompt space
            f"answer found in top-{k}": f"{found}/{len(EVAL)}", "mean top similarity": round(sum(search(kb, q, 1)[0][0] for q, _ in EVAL) / len(EVAL), 2)}

pd.DataFrame({name: evaluate(kb) for name, kb in KBS.items()}).T

,chunks,avg chars,context per question,answer found in top-3,mean top similarity
tiny,6412,96,288,6/14,0.68
medium,2115,293,879,10/14,0.63
huge,226,2757,8271,9/14,0.49
sections,1160,546,1638,10/14,0.62


Read the table across: **tiny** chunks score the *highest similarity* yet find the *fewest answers* — the matching phrase is there but the number next to it fell into the neighboring chunk. **huge** chunks put about 8,000 characters into every prompt, nine times medium's, and still miss answers because one vector now stands for many topics. **medium** and **sections** find about the same number of answers (in our test run 10 of 14 each — on 14 questions, one more or less is noise) with a fraction of huge's context, but not the *same* answers: sections found the meal limit, medium did not. None of this is a law of nature: **the right chunking depends on your documents and your questions — measure it like this, do not copy it.**

The answer texts have to be specific. Our first version checked for `"4%"`, `"$50"` and `"3 years"`: `"4%"` also matches *0.54%* in a fee list, `"$50"` a *$50 gift card*, and `"3 years"` a dozen other rows of the retention table. Loose answer texts make every strategy look better than it is.

We continue with `medium` — it is a tie, and section 7 shows what medium's miss on the meal question costs.

In [ ]:
kb = KBS["medium"]

## 6 Metadata filters — change *which documents compete*

Similarity search ranks chunks; a `where` filter first decides **which chunks are allowed to compete**. Metadata is exact and cheap, so use it whenever the question implies it: the user's country, a date range, a department.

In [18]:
q = "Which private medical insurance provider do I get?"
print("no filter   :", [(s, m["title"][:28]) for s, m, _ in search(kb, q)])              # every country's page competes
print("country=UK  :", [(s, m["title"][:28]) for s, m, _ in search(kb, q, where={"country": "UK"})])
print("country=IE  :", [(s, m["title"][:28]) for s, m, _ in search(kb, q, where={"country": "IE"})])

no filter   : [(0.53, 'GitLab Inc (US) Benefits'), (0.52, 'PTY LTD Benefits'), (0.49, 'GitLab LTD (UK) Benefits')]
country=UK  : [(0.49, 'GitLab LTD (UK) Benefits'), (0.48, 'GitLab LTD (UK) Benefits'), (0.46, 'GitLab LTD (UK) Benefits')]
country=IE  : [(0.49, 'GitLab Ireland Ltd'), (0.41, 'GitLab Ireland Ltd'), (0.4, 'GitLab Ireland Ltd')]


In [19]:
q = "How quickly must I submit an expense after buying something?"
print("all pages              :", [(s, m["title"][:26], m["updated"]) for s, m, _ in search(kb, q)])
print("updated since June 2026:", [(s, m["title"][:26], m["updated"]) for s, m, _ in search(kb, q, where={"updated": {"$gte": 20260601}})])
print("Finance only           :", [(s, m["title"][:26], m["updated"]) for s, m, _ in search(kb, q, where={"department": "Finance"})])
print("department=Legal       :", [(s, m["title"][:26]) for s, m, _ in search(kb, "Can I install Chrome extensions on my laptop?", where={"department": "Legal"})])

all pages              : [(0.52, 'Navan Purchasing Card', 20260827), (0.51, 'Spending Company Money', 20250130), (0.51, 'GitLab Offboarding FAQ', 20260728)]
updated since June 2026: [(0.52, 'Navan Purchasing Card', 20260827), (0.51, 'GitLab Offboarding FAQ', 20260728), (0.47, 'Global Travel and Expense ', 20260831)]
Finance only           : [(0.52, 'Navan Purchasing Card', 20260827), (0.51, 'Spending Company Money', 20250130), (0.47, 'Global Travel and Expense ', 20260831)]
department=Legal       : [(0.07, 'Policies related to Gifts '), (0.07, 'Anti-Retaliation Policy'), (0.06, 'Anti-Retaliation Policy')]


Four effects worth remembering: the **country filter turns an ambiguous question into a precise one** (without it, the US and Australian pages outrank the UK's); a **"recent pages only" filter silently dropped the best page** ("Spending Company Money" was last edited in January 2025) — filters hide as well as focus, so derive them from what the user actually needs; the **Finance filter changed nothing** here, because the best chunks already came from Finance pages — a filter only helps when the wrong documents were competing; and the **wrong department filter leaves only irrelevant chunks** with tiny similarity — which is exactly what the gate in the next section catches.

## 7 Generate a grounded answer — the whole pipeline in one function

Retrieve (with optional filter) → check the best similarity (refuse junk) → paste the chunks into the prompt as numbered passages → ask for citations → append the real sources.

In [20]:
def answer(question, k=3, where=None, threshold=0.40, search_for=None):
    hits = search(kb, search_for or question, k, where)                     # search_for: search with another text (Exercise 1)
    if not hits or hits[0][0] < threshold:                                   # nothing relevant -> do not let the model improvise
        return f"I don't have that information in the handbook. (best similarity {hits[0][0] if hits else 0})"
    context = "\n\n".join(f"[{i}] ({m['title']}) {text}" for i, (_, m, text) in enumerate(hits, 1))
    reply = chat([{"role": "system", "content": "Answer ONLY from the context passages. Cite them like [1]. If the context does not contain the answer, say so."},
                  {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    sources = "\n".join(f"[{i}] {m['title']} — {m['url']}" for i, (_, m, _) in enumerate(hits, 1))
    return f"{reply}\n\nSources:\n{sources}"

print(answer("What is the daily limit for meals while traveling for work?"))

The daily limit for meals while traveling for work is $120 USD (or local equivalent currency) per day [1].

Sources:
[1] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[2] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[3] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/




Three more questions: one the documents answer (with a country filter), one they do not, and one with the wrong filter.

In [22]:
print(answer("Which private medical insurance provider do I get, and who pays the premium?", where={"country": "IE"}), "\n")
print(answer("Who won the 2022 World Cup?"), "\n")                                                    # out of scope -> refused, not invented
print(answer("Can I install Chrome extensions on my laptop?", where={"department": "Legal"}))          # wrong filter -> refused, not invented

You get private medical insurance through **VHI**, and **GitLab** pays 100% of the premiums for you and your eligible dependents [2][3].

Sources:
[1] GitLab Ireland Ltd — https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/gitlab-ireland-ltd/
[2] GitLab Ireland Ltd — https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/gitlab-ireland-ltd/
[3] GitLab Ireland Ltd — https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/gitlab-ireland-ltd/ 

I don't have that information in the handbook. (best similarity 0.16) 

I don't have that information in the handbook. (best similarity 0.07)


Compare with step 0: same model, now sourced and refusing what it cannot back up — but only as right as the passages it receives. **Where does the 0.40 threshold in `answer()` come from?** From measuring, not guessing: the best similarity for the 14 questions above against four questions the handbook cannot answer.

In [23]:
OUT_OF_SCOPE = ["Who won the 2022 World Cup?", "How do I fix a flat bicycle tire?", "Best pizza recipe?", "What is the capital of France?"]
print("in scope  :", sorted(search(kb, q, 1)[0][0] for q, _ in EVAL))
print("out scope :", sorted(search(kb, q, 1)[0][0] for q in OUT_OF_SCOPE))

in scope  : [0.45, 0.47, 0.52, 0.57, 0.58, 0.59, 0.62, 0.63, 0.63, 0.66, 0.69, 0.71, 0.77, 0.77]
out scope : [0.16, 0.17, 0.23, 0.3]


In our test run the in-scope questions scored 0.43–0.79 and the out-of-scope ones 0.17–0.35, so the threshold goes in the gap: 0.40. That is 18 questions and a narrow gap. *"The capital of France"* scores 0.35 because a French remote-work policy exists; with a threshold of 0.35 it would pass the gate (0.35 is not below 0.35). The gate also cannot tell a relevant chunk from a merely similar one — the $80 passages passed it easily. Thresholds catch junk, not every wrong match, and they change with the embedding model, the chunking and your documents: re-measure whenever any of them changes.

## 8 Exercises: three prompts

Each exercise is a prompt in a code cell: replace *Write your prompt here*, keep the `{placeholders}`, run the cell, read the output.

### Exercise 1: Search with an imagined answer

 let the model write what an answer **would look like** in a handbook, and search with the question plus that imagined passage (the idea behind *HyDE*, Gao et al., 2022, *Precise Zero-Shot Dense Retrieval without Relevance Labels*). The imagined facts may be wrong — only their wording is used for the search; the answer still comes from the real chunks.

Write the prompt; keep `{question}`. The cell measures your search text on the 14 questions from section 5, prints the questions it gained or lost, and asks the meal question again.

In [27]:
SEARCH_PROMPT = '''Write two sentences that a company's employee handbook could contain to answer the question below. Write them as policy text, not as a reply to a person.
Question: {question}'''


passages = chat_batch([SEARCH_PROMPT.replace("{question}", q) for q, _ in EVAL], max_new_tokens=500)
imagined = {q: p for (q, _), p in zip(EVAL, passages)}
before = [any(has_answer(t, a) for _, _, t in search(kb, q)) for q, a in EVAL]
after = [any(has_answer(t, a) for _, _, t in search(kb, q + "\n" + imagined[q])) for q, a in EVAL]
print(f"answer in top-3:  question only {sum(before)}/{len(EVAL)}   question + imagined passage {sum(after)}/{len(EVAL)}\n")
for (q, _), b, a in zip(EVAL, before, after):
    if a != b:
        print(f"{'GAINED' if a else 'LOST  '} {q}\n       imagined: {imagined[q][:200]!r}")

meal = "What is the daily limit for meals while traveling for work?"
print("\n" + answer(meal, search_for=meal + "\n" + imagined[meal]))

answer in top-3:  question only 10/14   question + imagined passage 10/14


The daily limit for meals while traveling for work is $120 USD (or local equivalent currency) per day [1].

Sources:
[1] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[2] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[3] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/


### Exercise 2: A filter from the question

In section 6 a country filter turned *"Which private medical insurance provider do I get?"* into a precise question — but you typed the filter by hand. Let the model read it from the question: write a prompt that returns JSON `{"country": "IE"}` when the question names or implies one of the handbook's countries (UK, IE, NL, US, AU, CA, FR; a city counts) and `{"country": null}` when it does not. Keep `{question}`. The cell checks your prompt on ten questions whose country we know, then answers the first one with the filter it found.

In [28]:
COUNTRY_PROMPT = '''Which country's policies does this question need? The handbook has country pages for: UK (United Kingdom), IE (Ireland), NL (Netherlands), US (United States), AU (Australia), CA (Canada), FR (France).
If the question names or implies one of these countries (a city counts), reply with JSON like {"country": "IE"}. If it does not, reply {"country": null}.

Question: {question}'''


def country_filter(question):
    "Your prompt -> a `where` filter for search(), or None (no filter) when the model finds no known country."
    country = chat_json(COUNTRY_PROMPT.replace("{question}", question), max_new_tokens=30).get("country")
    return {"country": country} if country in ("UK", "IE", "NL", "US", "AU", "CA", "FR") else None

COUNTRY_TESTS = [  # (question, the country filter it needs; None = no filter)
    ("I'm based in Dublin. Which private medical insurance provider do I get?", "IE"),
    ("Which private medical insurance provider do I get?", None),
    ("How much does GitLab put into my pension? I work for the UK entity.", "UK"),
    ("As a team member in the Netherlands, how many vacation days do I get?", "NL"),
    ("What is the 401(k) match for team members in the United States?", "US"),
    ("I live in Sydney. What support do I get for private health insurance?", "AU"),
    ("What is the minimum password length?", None),
    ("Who pays for my internet when I work from home in Lyon?", "FR"),
    ("I work from Toronto. Does GitLab top up parental leave pay?", "CA"),
    ("What is the daily limit for meals while traveling for work?", None),
]
right = 0
for q, expected in COUNTRY_TESTS:
    f = country_filter(q)
    got = f["country"] if f else None
    right += got == expected
    print(f"{'ok' if got == expected else 'XX'}  got {str(got):<5} expected {str(expected):<5} | {q}")
print(f"{right}/{len(COUNTRY_TESTS)} correct\n")

q = COUNTRY_TESTS[0][0]
print(answer(q, where=country_filter(q)))

ok  got IE    expected IE    | I'm based in Dublin. Which private medical insurance provider do I get?
ok  got None  expected None  | Which private medical insurance provider do I get?
ok  got UK    expected UK    | How much does GitLab put into my pension? I work for the UK entity.
ok  got NL    expected NL    | As a team member in the Netherlands, how many vacation days do I get?
ok  got US    expected US    | What is the 401(k) match for team members in the United States?
ok  got AU    expected AU    | I live in Sydney. What support do I get for private health insurance?
ok  got None  expected None  | What is the minimum password length?
ok  got FR    expected FR    | Who pays for my internet when I work from home in Lyon?
ok  got CA    expected CA    | I work from Toronto. Does GitLab top up parental leave pay?
ok  got None  expected None  | What is the daily limit for meals while traveling for work?
10/10 correct

Based on the provided context, the private medical insurance provid

**What to look for.** In our test run the prompt got 10 of 10, and the filtered search answered the Dublin question from the Ireland page (VHI). A stricter variant we tried (*"do not guess"*) missed Lyon → FR — the same trade-off as the similarity gate, between missing a filter and inventing one. "No country" must mean **no filter**, not `{"country": "global"}`: that would hide every country page.

### Exercise 3: Follow-up questions

`answer()` has no memory: every call starts from zero (Day 1). Ask about the UK pension, then *"And in the Netherlands?"* — on its own the follow-up has no topic, so retrieval finds whatever Netherlands chunk is closest to "Netherlands". The usual fix is one `chat()` call that rewrites the follow-up into a **standalone question** before retrieval. Write that prompt; keep `{previous}` and `{follow_up}`.

In [30]:
REWRITE_PROMPT = '''Rewrite the follow-up question as a standalone question that someone can understand without the conversation: fill in what the follow-up leaves out, taken from the previous question. Keep the previous question's wording where it still applies. Reply with the question only.

Previous question: {previous}
Follow-up: {follow_up}'''


previous, follow_up = "How much does GitLab contribute to my pension in the UK?", "And in the Netherlands?"
print("PREVIOUS:", previous, "\n" + answer(previous), "\n")
print("FOLLOW-UP AS ASKED:", follow_up, "\n" + answer(follow_up), "\n")
standalone = chat(REWRITE_PROMPT.replace("{previous}", previous).replace("{follow_up}", follow_up), max_new_tokens=100)
print("STANDALONE:", standalone, "\n" + answer(standalone))

PREVIOUS: How much does GitLab contribute to my pension in the UK? 
GitLab contributes 4% of your basic salary to the pension scheme [1][2].

Sources:
[1] GitLab LTD (UK) Benefits — https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/ltd-benefits-uk/
[2] GitLab LTD (UK) Benefits — https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/ltd-benefits-uk/
[3] GitLab LTD (UK) Benefits — https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/ltd-benefits-uk/ 

FOLLOW-UP AS ASKED: And in the Netherlands? 
In the Netherlands, full-time team members receive a EUR 150 gross monthly medical allowance to help cover healthcare costs [1]. They must also purchase their own basic health insurance (basisverzekering) as required by Dutch law, as not having it can result in fines from the Sociale Verzekeringsbank [1]. Additionally, team members can transfer pension value from their former employer within

**What to look for.** In our test run the follow-up as asked passed the gate and came back with an answer about vacation money and health insurance — Netherlands chunks, wrong topic. The rewritten question, *"How much does GitLab contribute to my pension in the Netherlands?"*, retrieved the Dutch pension chunk and the right answer: 3%. The rewrite only helps if the standalone question retrieves well: when we started from section 5's wording (*"What percentage of my salary does the company pay into the UK pension?"*), medium's top 3 missed the employer's share in both countries, and the UK answer said 5% — the team member's share. Retrieval again.

## Summary

- **Chunking is measured, not chosen by taste**: on these 14 questions medium and sections tie, tiny loses the numbers, huge costs nine times the context. Answer texts must be specific, or the measurement lies.
- **Metadata filters** decide which chunks may compete: they focus (country) and they hide (date). Derive them from what the user needs.
- **The similarity gate** catches junk, at a threshold measured in the gap between in-scope and out-of-scope questions — not every wrong match.
- **A grounded answer is only as right as its passages.** When an answer is wrong, print `search(kb, q)` first: the $80 answer was a retrieval problem, not a model problem.
- **Changing what you search with** (Exercise 1) fixed the meal question and broke two others: judge a retrieval change on the whole question set, not on the example that motivated it.
- `answer()` has **no memory**: follow-ups need the conversation turned into a standalone question.

| Component | In this lab | Production upgrade |
|---|---|---|
| Loader + metadata | `fetch()` from the handbook API | connectors (SharePoint, Confluence, tickets), access control in metadata |
| Cleaner | `clean()` — 3 regexes | format-aware parsers (PDF layout, tables) |
| Chunker | `chunk_fixed` / `chunk_sections` + **measured** | parent–child chunks, overlap, per-document-type rules |
| Embeddings + vector DB | `all-MiniLM-L6-v2` + ChromaDB | bigger embedder, hybrid keyword + vector search, reranker |
| Retriever | `search()` with `where` filters | filters from the user's identity and question (country, date, permissions) |
| Generator + gate | `answer()` with a measured threshold | evaluation set: hit rate, faithfulness, answer quality |

*Handbook text © GitLab Inc., [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).* **Next → Day2_RAG_Evaluation:** turn the 14-question check into a real evaluation (hit rate, MRR, faithfulness). Tool calling and agents come on Day 3.

---
# Extras

Four optional experiments. They all use the GitLab handbook collection (`kb = KBS["medium"]`), except the last one, which switches `answer()` to your own documents.

## Extra A: Overlap: does it rescue what tiny lost?

Overlapping chunks repeat the end of the previous chunk at the start of the next, so a sentence cut at a boundary appears whole in one of the two. `chunk_overlap(text, size, overlap)` makes chunks of about `size` characters whose first `overlap` characters are the last `overlap` characters of the previous chunk. Here: tiny 100-character chunks with 50 characters of overlap — twice as many chunks as tiny.

In [31]:
def chunk_overlap(text, size=300, overlap=60):
    """Chunks of about `size` characters; each starts with the last `overlap` characters of the previous chunk."""
    pieces = chunk_fixed(text, size - overlap)                       # the new text of each chunk
    return [pieces[0]] + [prev[-overlap:] + " " + p for prev, p in zip(pieces, pieces[1:])]

KBS["tiny+overlap"] = index("tiny_overlap", lambda t: chunk_overlap(t, 100, 50))
pd.DataFrame({name: evaluate(KBS[name]) for name in ["tiny", "tiny+overlap", "medium"]}).T

,chunks,avg chars,context per question,answer found in top-3,mean top similarity
tiny,6412,96,288,6/14,0.68
tiny+overlap,13140,93,279,9/14,0.72
medium,2115,293,879,10/14,0.63


In our test run overlap raised tiny from 6 to 9 answers of 14 — it rescues cut sentences, at the price of twice the chunks to embed and store — but it still did not beat medium.

## Extra B: How many chunks? k = 1, 3, 6

More passages raise the chance that the answer is among them, and make the prompt longer and noisier. Count it, then ask the meal question again with k = 6 (the $120 chunk ranked 6th):

In [32]:
rows = {}
for k in [1, 3, 6]:
    r = evaluate(kb, k)
    rows[f"k={k}"] = {"context per question": r["context per question"], "answer found": r[f"answer found in top-{k}"]}
display(pd.DataFrame(rows).T)
print(answer("What is the daily limit for meals while traveling for work?", k=6))

,context per question,answer found
k=1,546,8/14
k=3,1638,10/14
k=6,3276,14/14


The daily limit for meals while traveling for work is $120 USD (or local equivalent currency) per day [1].

Sources:
[1] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[2] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[3] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[4] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[5] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[6] GitLab France S.A.S. Remote Work Charter — https://handbook.gitlab.com/handbook/people-policies/france-sas/remote-work-charter/


In our test run k = 1 found 8 answers, k = 3 found 10 and k = 6 found 14 of 14, for twice the context.

## Extra C: Keyword + vector

Embeddings match meaning; some questions need an exact word — a product name, an error code, the name of a record type. ChromaDB can require a word with `where_document={"$contains": ...}` (case-sensitive) and rank only the chunks that contain it by similarity. In section 5 no strategy found the retention period of production changes:

In [34]:
q = "How long are production change records retained?"
print("vector only              :", [(s, text[:80]) for s, _, text in search(kb, q)])
r = kb.query(query_texts=[q], n_results=3, where_document={"$contains": "Production changes"})   # only chunks with these exact words
print("must contain the keywords:", [(round(1 - d, 2), text[:200]) for text, d in zip(r["documents"][0], r["distances"][0])])
print("contains the answer      :", any("Production changes | 3 years" in text for text in r["documents"][0]))

vector only              : [(0.45, '## Retention & Disposal Requirements Procedure\nanalysis records | 3 years | [GCP'), (0.39, '### Record Retention All documents related to the background check process must '), (0.39, '### For leaves 31 days or less - No changes will be made to the team member’s jo')]
must contain the keywords: [(0.36, '## Retention & Disposal Requirements Procedure | Record | Retention Requirement | Disposal Requirement | |----------------------------------------------------|---------------------------|-------------')]
contains the answer      : True


## Extra D: 🗂️ Your own documents

Upload 1–5 files (`.pdf .txt .md .csv`) from your organization — a policy, a report, a manual. They stay inside this Colab session. The same clean → chunk → embed → search → answer pipeline runs on them; you become the source of truth for whether the answers are right.

In [ ]:
from google.colab import files
from pypdf import PdfReader

def upload():
    """Open the file picker; return {filename: text} for .pdf / .txt / .md / .csv files."""
    texts = {}
    for name, blob in files.upload().items():
        if name.lower().endswith(".pdf"):
            open(name, "wb").write(blob)
            texts[name] = "\n".join(page.extract_text() or "" for page in PdfReader(name).pages)
        else:
            texts[name] = blob.decode("utf-8", errors="ignore")
    print({k: f"{len(v)} chars" for k, v in texts.items()})
    return texts

In [ ]:
mine = upload()

client.delete_collection("my_documents") if "my_documents" in [c.name for c in client.list_collections()] else None
my_kb = client.create_collection("my_documents", embedding_function=embedder, metadata={"hnsw:space": "cosine"})
for name, text in mine.items():
    pieces = chunk_sections(clean(text))
    my_kb.add(ids=[f"{name}-{i}" for i in range(len(pieces))], documents=pieces,
              metadatas=[{"title": name, "department": "mine", "country": "global", "updated": 20260907, "url": name}] * len(pieces))
print(my_kb.count(), "chunks from your files")

In [ ]:
kb = my_kb                                                                       # point answer() at your documents
while True:                                                                      # ask anything, 'quit' to stop
    q = input("Ask your documents: ").strip()
    if q.lower() in ("quit", "exit", "q", ""): break
    print(answer(q), "\n")

If a question you *know* is answered comes back as "I don't have that information", the best similarity was below the threshold: your wording differs from the document's. Rephrase, or try `answer(q, threshold=0.25)`. If the answer is wrong, print `search(kb, q)` first — usually the retrieval, not the model, is the problem (remember the $80 answer). To go back to the handbook: `kb = KBS["medium"]`.